# 04. 스트리밍과 content blocks

응답 조각은 어떤 순서로 도착하고 최종 텍스트와 어떻게 연결될까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 스트리밍 실습의 고정 패키지 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 API 키의 존재 여부만 확인합니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 스트리밍에도 같은 모델 ID와 초 단위 타임아웃·출력 상한을 적용합니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## 스트리밍 호출

`stream()`이 반환하는 `AIMessageChunk`를 도착 순서대로 저장합니다.

In [4]:
# stream이 내보내는 AIMessageChunk를 도착 순서대로 chunks 목록에 저장합니다.
messages = [("user", "LangChain 스트리밍을 열두 단어 이내로 설명하세요.")]
chunks = []
for chunk in model.stream(messages):
    chunks.append(chunk)
    print(chunk.text, end="", flush=True)
print()
print("조각 수:", len(chunks))

실

시간

으로

 생성

되는

...

하는

 기능

입니다

.


조각 수: 20


## 조각 결합

`+` 연산으로 메시지 조각을 결합하고 최종 텍스트를 확인합니다.

In [5]:
# 청크의 + 연산으로 메타데이터를 보존한 최종 AIMessageChunk를 만듭니다.
combined = chunks[0]
for chunk in chunks[1:]:
    combined = combined + chunk
print("결합 텍스트:", combined.text)
print("결합 성공:", bool(combined.text.strip()))

결합 텍스트: 실시간으로 생성되는 응답을 부분별로 전달하는 기능입니다.
결합 성공: True


## content blocks 관찰

각 조각의 표준 콘텐츠 블록 수를 세되 제공자별 블록 모양을 단정하지 않습니다.

In [6]:
# 각 청크의 표준 content_blocks 개수와 합계를 관찰값으로 남깁니다.
block_counts = [len(chunk.content_blocks) for chunk in chunks]
print("조각별 블록 수:", block_counts)
print("전체 블록 수:", sum(block_counts))
representative = next((c.content_blocks[0] for c in chunks if c.content_blocks), None)
print("대표 content block:", representative)
print("결합 메시지 usage 있음:", combined.usage_metadata is not None)

조각별 블록 수: [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
전체 블록 수: 16
대표 content block: {'type': 'text', 'text': '실'}
결합 메시지 usage 있음: True


## invoke와 결과 비교

같은 질문을 일반 호출로 한 번 더 실행해 반환 타입을 비교합니다.

In [7]:
# 같은 질문의 invoke 반환 타입과 결합 청크 타입을 비교합니다.
single = model.invoke(messages)
print("stream 반환 타입:", type(chunks[0]).__name__)
print("invoke 반환 타입:", type(single).__name__)
print("두 결과 모두 텍스트 있음:", bool(combined.text and single.text))

stream 반환 타입: AIMessageChunk
invoke 반환 타입: AIMessage
두 결과 모두 텍스트 있음: True
